# 03 · Inputs for the three strategy legs

This notebook prepares a shared event register, a month-end candidate universe for anticipation, and daily event windows for execution analysis. The post-inclusion notebook can use the same addition keys to compare its monthly holdings with the event register.

Run from the repository root or `notebooks/`. The monthly candidate build reads the CRSP monthly export. Set `build_daily_windows = True` to scan the roughly 10 GB daily export. The prepared CSV files are written to the Git-ignored `Data/prepared/` directory.

In [ ]:
from pathlib import Path
from tempfile import NamedTemporaryFile

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd

build_daily_windows = False
save_prepared = True

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"
with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

prepared_dir = data_dir / "prepared"


def save_csv(table, filename):
    if save_prepared:
        prepared_dir.mkdir(parents=True, exist_ok=True)
        with NamedTemporaryFile(
            mode="w", encoding="utf-8", newline="", dir=prepared_dir, suffix=".tmp", delete=False
        ) as temp_file:
            temp_path = Path(temp_file.name)
            table.to_csv(temp_file, index=False)
        temp_path.replace(prepared_dir / filename)

## Shared addition dates and source evidence

An addition is keyed by `permno, effective_date`. The register keeps the supplied announcement and migration fields separate from evidence in the S&P notice seed. A documented MidCap 400 exit within five calendar days is marked as a migration. Other events retain an unknown verified classification.

The next trading session after a dated announcement is a date-only proxy. When an S&P notice confirms a change before the effective-day open, the preceding session is the reference for the execution close. The daily CRSP price is a proxy for that close. Auction fills require separate execution data. For the 2025 Solstice and Qnity spin-offs, the notice's addition and replacement dates are stored separately.

In [ ]:
notices = public_notices.drop(columns=["company", "event_type"]).rename(
    columns={"announcement_date": "verified_announcement_date"}
)
if notices["verified_announcement_date"].gt(notices["effective_date"]).any():
    raise ValueError("An S&P notice date is later than its effective date.")

event_register = events.rename(columns={
    "announcement_date": "reported_announcement_date",
    "migration_from_midcap400": "reported_midcap400_migration",
}).merge(notices, on=notice_key, how="left", validate="many_to_one")
if event_register["source_url"].notna().sum() != len(notices):
    raise ValueError("Each S&P notice seed row must match one addition in the local register.")
event_register["announcement_date"] = event_register["verified_announcement_date"].combine_first(
    event_register["reported_announcement_date"]
)
event_register["verified_midcap400_migration"] = pd.Series(
    pd.NA, index=event_register.index, dtype="boolean"
)
migration_gap = (event_register["effective_date"] - event_register["midcap400_exit_date"]).dt.days
event_register.loc[migration_gap.abs().le(5), "verified_midcap400_migration"] = True

trading_days = pd.DatetimeIndex(calendar_dates["index_daily"]).sort_values()
if not trading_days.is_unique:
    raise ValueError("The daily benchmark calendar has repeated dates.")
session_by_date = pd.Series(np.arange(len(trading_days)), index=trading_days)
date_by_session = pd.Series(trading_days)
event_register["effective_session_number"] = event_register["effective_date"].map(session_by_date)
if event_register["effective_session_number"].isna().any():
    raise ValueError("An effective date falls outside the trading calendar. Check the event source.")
event_register["previous_session"] = (
    event_register["effective_session_number"] - 1
).map(date_by_session)
event_register["verified_execution_close"] = event_register["previous_session"].where(
    event_register["effective_timing"].eq("before_open")
)
has_announcement = event_register["announcement_date"].notna()
announcement_session_numbers = pd.Series(
    trading_days.searchsorted(event_register.loc[has_announcement, "announcement_date"], side="right"),
    index=event_register.index[has_announcement],
)
event_register["announcement_next_session_proxy"] = announcement_session_numbers.map(date_by_session)
event_register["first_post_inclusion_month"] = event_register["effective_date"].dt.to_period("M") + 1

save_csv(event_register.drop(columns="effective_session_number"), "strategy_events.csv")
display(pd.Series({
    "Provisional additions": len(event_register),
    "Announcement dates available": int(has_announcement.sum()),
    "Additions checked against S&P": int(event_register["source_url"].notna().sum()),
    "Execution close reference dates": int(event_register["verified_execution_close"].notna().sum()),
    "Documented MidCap 400 migrations": int(event_register["verified_midcap400_migration"].sum()),
}, name="event coverage"))

## Reconstruct the historical S&P 500 universe

Use the initial workbook and the historical spells, treating both interval endpoints as inclusive. Cap the workbook intervals at the historical export boundary. This yields a provisional month-end reconstruction through December 2022. Extending the candidate universe through 2025 requires a reconciled membership history, including exits.

The candidate screen is security-level. It uses the CRSP classification valid at formation for U.S.-incorporated common equity on NYSE, NYSE American, or Nasdaq. Securities flagged as delisted in the formation month are excluded. Market capitalization is CRSP's security-level market value. It serves as a size proxy for the eventual float-adjusted screen.

In [ ]:
membership_key = ["permno", "formation_date"]
formation_calendar = pd.DataFrame({
    "formation_date": pd.DatetimeIndex(calendar_dates["index_monthly"]).sort_values()
})
formation_calendar = formation_calendar.loc[formation_calendar["formation_date"].le(history_end)]
snapshot_spells = snapshot.rename(columns={
    "PERMNO": "permno", "SP500 Start": "start", "SP500 End": "end",
})[["permno", "start", "end"]].copy()
snapshot_spells["end"] = snapshot_spells["end"].clip(upper=history_end)
membership_spells = pd.concat(
    [historical_spells[["permno", "start", "end"]], snapshot_spells], ignore_index=True
).drop_duplicates()

membership_grid = membership_spells.merge(formation_calendar, how="cross")
member_months = membership_grid.loc[
    membership_grid["formation_date"].between(membership_grid["start"], membership_grid["end"]),
    membership_key,
].drop_duplicates()
del membership_grid

security_info = pd.read_csv(
    paths["crsp_names"],
    usecols=[
        "permno", "secinfostartdt", "secinfoenddt", "usincflg",
        "securitytype", "securitysubtype", "sharetype", "primaryexch", "siccd",
    ],
    parse_dates=["secinfostartdt", "secinfoenddt"],
)
eligible_security_info = security_info.loc[
    security_info["usincflg"].eq("Y") & security_info["securitytype"].eq("EQTY")
    & security_info["securitysubtype"].eq("COM") & security_info["sharetype"].eq("NS")
    & security_info["primaryexch"].isin(["N", "A", "Q"]),
    ["permno", "secinfostartdt", "secinfoenddt", "primaryexch", "siccd"],
].sort_values("secinfostartdt")

## Leg 1 · Anticipation candidates and preliminary labels

The feature table uses observations available by each formation date. It ranks nonmembers by security-level market capitalization and keeps a liquidity proxy based on month-end price and monthly share volume.

The separate label table records effective inclusion in the next three months, with missing labels for incomplete horizons. It is useful for checking the pipeline. Testing a prediction made before public announcement requires announcement-date labels and coverage checks. The full model also needs historical float, earnings availability dates, and MidCap 400 membership. A Compustat fiscal period end alone does not establish when earnings became public.

In [ ]:
market_columns = {
    "PERMNO": "permno", "MthCalDt": "formation_date", "MthCap": "market_cap_thousands",
    "MthPrc": "month_end_price", "MthVol": "monthly_share_volume",
}
candidate_chunks = []
for raw_chunk in pd.read_csv(
    paths["crsp_monthly"], usecols=[*market_columns, "MthDelFlg"],
    parse_dates=["MthCalDt"], chunksize=500_000,
):
    chunk = raw_chunk.rename(columns=market_columns)
    chunk = chunk.loc[
        chunk["formation_date"].isin(formation_calendar["formation_date"])
        & chunk["market_cap_thousands"].gt(0)
        & chunk["MthDelFlg"].eq("N")
    ].drop(columns="MthDelFlg").sort_values("formation_date")
    dated = pd.merge_asof(
        chunk, eligible_security_info,
        left_on="formation_date", right_on="secinfostartdt", by="permno", direction="backward",
    )
    candidate_chunks.append(
        dated.loc[dated["formation_date"].le(dated["secinfoenddt"])]
        .drop(columns=["secinfostartdt", "secinfoenddt"])
    )

candidate_features = pd.concat(candidate_chunks, ignore_index=True).drop_duplicates()
if candidate_features.duplicated(membership_key).any():
    raise ValueError("Conflicting monthly market fields for a candidate and formation date.")
candidate_features = candidate_features.merge(
    member_months.assign(in_sp500=True), on=membership_key, how="left", validate="one_to_one"
)
candidate_features = candidate_features.loc[candidate_features["in_sp500"].isna()].drop(columns="in_sp500")
candidate_features["market_cap_rank"] = (
    candidate_features.groupby("formation_date")["market_cap_thousands"]
    .rank(method="min", ascending=False).astype(int)
)
candidate_features["dollar_volume_proxy"] = (
    candidate_features["month_end_price"].abs() * candidate_features["monthly_share_volume"]
)
candidate_features = candidate_features.sort_values(
    ["formation_date", "market_cap_rank", "permno"], ignore_index=True
)
del candidate_chunks

historical_targets = event_register.loc[
    event_register["effective_date"].le(history_end) & event_register["permno"].notna()
    & ~event_register["likely_spin_or_merger"],
    ["permno", "effective_date"],
].astype({"permno": "int64"}).sort_values("effective_date")
candidate_labels = pd.merge_asof(
    candidate_features[membership_key].sort_values("formation_date", kind="stable"), historical_targets,
    left_on="formation_date", right_on="effective_date", by="permno",
    direction="forward", allow_exact_matches=False,
)
formation_month = candidate_labels["formation_date"].dt.to_period("M")
horizon_month = formation_month + 3
candidate_labels["effective_in_next_3_months"] = (
    candidate_labels.pop("effective_date").dt.to_period("M").le(horizon_month)
    .where(horizon_month.le(history_end.to_period("M"))).astype("Int8")
)

save_csv(candidate_features, "anticipation_candidates.csv")
save_csv(candidate_labels, "anticipation_effective_date_labels.csv")
display(pd.Series({
    "Candidate security-months": len(candidate_features),
    "Formation months": candidate_features["formation_date"].nunique(),
    "Three-month effective-date labels available": int(candidate_labels["effective_in_next_3_months"].count()),
}, name="anticipation coverage"))

## Leg 2 · Effective-date and announcement windows

Scan the daily export by chunks and retain event windows only. Windows start 100 sessions before the earlier of the effective session or available announcement proxy, and end 21 sessions after the effective date. The available calendar bounds the window. These inputs support an execution study of the successful candidates and a separate daily event study.

The table keeps daily returns, the S&P 500 series, prices, volume, and bid and ask quotes. Repeated rows that agree on the retained fields are collapsed. Conflicting observations and missing returns remain visible through the coverage column.

In [ ]:
daily_event_windows = None
if build_daily_windows:
    windows = []
    for event in event_register.loc[event_register["permno"].notna()].itertuples():
        effective = int(event.effective_session_number)
        announcement = session_by_date.get(event.announcement_next_session_proxy)
        first_anchor = min(effective, announcement) if announcement is not None else effective
        sessions = np.arange(max(0, first_anchor - 100), min(len(trading_days), effective + 22))
        windows.append(pd.DataFrame({
            "permno": int(event.permno),
            "effective_date": event.effective_date,
            "trade_date": trading_days[sessions],
            "relative_to_effective": sessions - effective,
            "relative_to_announcement_proxy": sessions - announcement if announcement is not None else pd.NA,
            "is_verified_execution_close": trading_days[sessions] == event.verified_execution_close,
        }))
    daily_event_windows = pd.concat(windows, ignore_index=True)
    daily_event_windows["trade_date"] = daily_event_windows["trade_date"].dt.strftime("%Y-%m-%d")
    daily_key = ["permno", "trade_date"]
    required_dates = daily_event_windows[daily_key].drop_duplicates()

    daily_columns = {
        "PERMNO": "permno", "DlyCalDt": "trade_date", "DlyRet": "stock_return",
        "DlyPrc": "price", "DlyVol": "share_volume", "DlyBid": "bid",
        "DlyAsk": "ask", "DlyDelFlg": "delist_flag",
    }
    event_ids = required_dates["permno"].unique()
    matched_chunks = []
    for raw_chunk in pd.read_csv(paths["crsp_daily"], usecols=list(daily_columns), chunksize=750_000):
        chunk = raw_chunk.rename(columns=daily_columns)
        matched_chunks.append(
            chunk.loc[chunk["permno"].isin(event_ids)].merge(
                required_dates, on=daily_key, how="inner", validate="many_to_one",
            )
        )

    quotes = pd.concat(matched_chunks, ignore_index=True).drop_duplicates()
    conflicts = quotes.duplicated(daily_key, keep=False)
    conflicting_keys = quotes.loc[conflicts, daily_key].drop_duplicates()
    daily_event_windows = daily_event_windows.merge(
        quotes.loc[~conflicts], on=daily_key, how="left", validate="many_to_one",
    ).merge(
        conflicting_keys.assign(conflicting_quotes=True),
        on=daily_key, how="left", validate="many_to_one",
    )
    daily_event_windows["coverage"] = np.select(
        [daily_event_windows.pop("conflicting_quotes").eq(True),
         daily_event_windows["stock_return"].isna()],
        ["conflicting_source_rows", "missing_return"], default="observed",
    )
    daily_benchmark = pd.read_csv(
        paths["index_daily"], usecols=["dlycaldt", "sprtrn"],
    ).rename(columns={"dlycaldt": "trade_date", "sprtrn": "sp500_price_return"})
    daily_event_windows = daily_event_windows.merge(
        daily_benchmark, on="trade_date", how="left", validate="many_to_one",
    )
    save_csv(daily_event_windows, "inclusion_daily_windows.csv")
    display(daily_event_windows["coverage"].value_counts())

## Leg 3 · Post-inclusion inputs

The shared event register contains the effective date, first eligible full holding month, and twelve-month calendar coverage. Use `permno, effective_date` to join these dates and source fields to the monthly CRSP holdings in `02_post_inclusion.ipynb`. That notebook tests a gross market-hedged short. The migration comparison and net returns after costs are further steps.

The cell below prepares the Fama–French five factors and momentum from the original archives. Returns are in decimal units. [Kenneth French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).

In [ ]:
five_factor_path = data_dir / "F-F_Research_Data_5_Factors_2x3_CSV.zip"
momentum_path = data_dir / "F-F_Momentum_Factor_CSV.zip"
monthly_factors = None
if five_factor_path.is_file() and momentum_path.is_file():
    five_factors = read_monthly_factors(five_factor_path, ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"])
    momentum = read_monthly_factors(momentum_path, ["Mom"])
    monthly_calendar = pd.PeriodIndex(calendar_dates["index_monthly"].dt.to_period("M"), name="month")
    monthly_factors = five_factors.join(momentum, validate="one_to_one").reindex(monthly_calendar)
    if monthly_factors.isna().any().any():
        raise ValueError("The monthly factors must cover the full CRSP monthly calendar.")
    save_csv(monthly_factors.reset_index(), "monthly_ff5_momentum.csv")
    print(f"Five-factor and momentum coverage: {len(monthly_factors)} months.")
else:
    print("Place the monthly five-factor and momentum archives in Data/ to prepare the multifactor inputs.")